In [6]:
import os
import numpy as np
import pandas as pd

os.chdir("/home/abid/binta/walkability_abid/")

from config.static_ws import VIF_FEATS_STATIC, STATIC_DATA_PATH, STATIC_WEIGHTS
from src.helpers import compute_vif

np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.float_format", "{:.3f}".format)


Print VIF STATISTICS

In [7]:
df = pd.read_csv(STATIC_DATA_PATH)
vif_static = compute_vif(df[VIF_FEATS_STATIC])
vif_static


drivable_inv_n           5.049
intersection_density_n   3.075
walkway_n                2.571
sidewalk_n               2.224
essential_poi_count_n    1.982
transit_proximity_n      1.933
poi_entropy_n            1.537
dtype: float64

you can (and should) perturb all feature weights at once by ±25% and see if rankings stay similar. It’s not “required,” but it’s a great sanity check when weights are subjective or you’ll share results.

Here’s a compact drop-in you can run after you’ve computed walkscore_static (or any composite built as a weighted sum of normalized feature columns). It does two things:

Global sensitivity: multiplies every weight by a random factor in [0.75, 1.25], re-normalizes, recomputes scores, and compares rankings.

One-at-a-time: perturbs each weight individually by ±25% to see which levers matter most.

In [19]:
FEATS = list(STATIC_WEIGHTS.keys())
W0 = pd.Series(STATIC_WEIGHTS, dtype=float)
W0 = W0 / W0.abs().sum()  # L1 normalize

X = df[FEATS].copy()
score_base = (X * W0).sum(axis=1)
rank_base  = score_base.rank(ascending=False, method="min").astype(int)

# ---- 1) Global ±25% perturbations on ALL weights ----
S = 200  # number of trials
rng = np.random.default_rng(123)
K = min(10, len(df))  # top-K overlap metric


spearmans, topk_overlap, abs_rank_mean, abs_rank_p90 = [], [], [], []
for _ in range(S):
    noise = rng.uniform(0.75, 1.25, size=len(W0))
    Wp = pd.Series(W0.values * noise, index=W0.index)
    Wp = Wp / Wp.abs().sum()  # re-normalize

    score_p = (X * Wp).sum(axis=1)
    rank_p  = score_p.rank(ascending=False, method="min").astype(int)

    # similarity metrics
    spearmans.append(score_base.corr(score_p, method="spearman"))
    top_base = set(score_base.nlargest(K).index)
    top_pert = set(score_p.nlargest(K).index)
    topk_overlap.append(len(top_base & top_pert) / float(K))
    dr = (rank_p - rank_base).abs()
    abs_rank_mean.append(float(dr.mean()))
    abs_rank_p90.append(float(dr.quantile(0.90)))

sens_global = pd.DataFrame({
    "spearman_rho_mean": [np.nanmean(spearmans)],
    "spearman_rho_p10":  [np.nanquantile(spearmans, 0.10)],
    "spearman_rho_p90":  [np.nanquantile(spearmans, 0.90)],
    f"top{K}_overlap_mean": [np.mean(topk_overlap)],
    f"top{K}_overlap_p10":  [np.quantile(topk_overlap, 0.10)],
    f"top{K}_overlap_p90":  [np.quantile(topk_overlap, 0.90)],
    "abs_rank_change_mean": [np.mean(abs_rank_mean)/ len(df)],
    "abs_rank_change_p90":  [np.mean(abs_rank_p90)/ len(df)],
}).round(3)

# ---- 2) One-at-a-time ±25% per feature ----
rows = []
for f in FEATS:
    Wp = W0.copy()
    Wp[f] *= 1.25
    Wp = Wp / Wp.abs().sum()
    s_up  = (X * Wp).sum(axis=1)
    r_up  = s_up.rank(ascending=False, method="min").astype(int)
    dr_up = (r_up - rank_base).abs()

    Wp = W0.copy()
    Wp[f] *= 0.75
    Wp = Wp / Wp.abs().sum()
    s_dn  = (X * Wp).sum(axis=1)
    r_dn  = s_dn.rank(ascending=False, method="min").astype(int)
    dr_dn = (r_dn - rank_base).abs()

    rows.append({
        "feature": f,
        "mean_abs_rank_change_up":   float(dr_up.mean() / len(df)),
        "p90_abs_rank_change_up":    float(dr_up.quantile(0.90) / len(df)),
        "mean_abs_rank_change_down": float(dr_dn.mean() / len(df)),
        "p90_abs_rank_change_down":  float(dr_dn.quantile(0.90) / len(df)),
        "spearman_up":               float(score_base.corr(s_up, method="spearman")),
        "spearman_down":             float(score_base.corr(s_dn, method="spearman")),
    })

sens_oaat = pd.DataFrame(rows).sort_values("p90_abs_rank_change_up", ascending=False).round(3)

In [20]:
sens_global

,spearman_rho_mean,spearman_rho_p10,spearman_rho_p90,top10_overlap_mean,top10_overlap_p10,top10_overlap_p90,abs_rank_change_mean,abs_rank_change_p90
0,0.996,0.993,0.998,0.970,0.900,1.000,0.017,0.041


In [17]:
sens_oaat

,feature,mean_abs_rank_change_up,p90_abs_rank_change_up,mean_abs_rank_change_down,p90_abs_rank_change_down,spearman_up,spearman_down
2,essential_poi_count_n,0.012,0.039,0.012,0.033,0.997,0.997
6,drivable_inv_n,0.012,0.033,0.012,0.029,0.998,0.998
4,walkway_n,0.011,0.029,0.012,0.032,0.998,0.998
0,intersection_density_n,0.011,0.029,0.011,0.027,0.998,0.998
3,transit_proximity_n,0.011,0.025,0.011,0.028,0.998,0.998
5,sidewalk_n,0.011,0.024,0.014,0.039,0.999,0.997
1,poi_entropy_n,0.011,0.024,0.011,0.024,0.999,0.998
